# Drug Data Collection and Dataset Preparation

## Objective

The objective of this notebook is to create a unified and clean dataset of drug reviews by combining training and test datasets. This dataset will serve as the foundation for NLP, machine learning, and deep learning tasks.

## Dataset Description

We use two datasets:

* drugsComTrain_raw.csv
* drugsComTest_raw.csv

Both datasets contain:

* drugName: name of the drug
* condition: medical condition
* review: patient feedback (text)
* rating: effectiveness score (1–10)
* date: review date
* usefulCount: usefulness score

## Approach

* Load both datasets
* Merge into a single dataset
* Clean and preprocess data
* Prepare features for downstream tasks

## Output

A clean and structured dataset ready for:

* NLP processing
* feature engineering
* modeling


In [16]:
import pandas as pd
import numpy as np

In [19]:
train_df = pd.read_csv("../../Drug_module/data/drug_review_train.csv")
test_df = pd.read_csv("../../Drug_module/data/drug_review_train.csv")


print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)


Train Shape: (110811, 9)
Test Shape: (110811, 9)


## Combining Both the datasets


In [20]:
df = pd.concat([train_df, test_df], axis=0)

df.reset_index(drop=True, inplace=True)

print("Combined Shape:", df.shape)

Combined Shape: (221622, 9)


# Data Inspection

We inspect the combined dataset to understand its structure, data types, and missing values.


In [21]:
df.head()

df.info()

df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 221622 entries, 0 to 221621
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     221622 non-null  int64  
 1   patient_id     221622 non-null  int64  
 2   drugName       221622 non-null  str    
 3   condition      221622 non-null  str    
 4   review         221622 non-null  str    
 5   rating         221622 non-null  float64
 6   date           221622 non-null  str    
 7   usefulCount    221622 non-null  int64  
 8   review_length  221622 non-null  int64  
dtypes: float64(1), int64(4), str(4)
memory usage: 15.2 MB


Unnamed: 0       0
patient_id       0
drugName         0
condition        0
review           0
rating           0
date             0
usefulCount      0
review_length    0
dtype: int64

In [22]:
df.head()

,Unnamed: 0,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,0,89879,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,"April 20, 2013",69,147
1,1,143975,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,"August 7, 2016",4,136
2,2,106473,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,"May 11, 2016",6,140
3,3,184526,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,"March 19, 2012",124,104
4,4,91587,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,"August 1, 2010",101,74


In [23]:
df = df[['drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']]

In [26]:
df.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
221617     True
221618     True
221619     True
221620     True
221621     True
Length: 221622, dtype: bool

## REMOVING DUPLICATED RECORDS


In [27]:
df = df.drop_duplicates()

# Text Cleaning

The review text is cleaned to remove noise such as special characters and inconsistent casing.


In [28]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_review'] = df['review'].apply(clean_text)

# Date Processing

The date column is converted into datetime format and used to extract time-based features.


In [30]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

In [31]:
df.head()

,drugName,condition,review,rating,date,usefulCount,clean_review,year,month
0,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,2013-04-20,69,i have used restasis for about a year now and ...,2013,4
1,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,2016-08-07,4,my experience has been somewhat mixed i have b...,2016,8
2,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,2016-05-11,6,this is my second implanon would not recommend...,2016,5
3,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,2012-03-19,124,i recommend taking as prescribed and the bottl...,2012,3
4,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,2010-08-01,101,i have been on ampyra for days and have been ...,2010,8


In [32]:
df['review_length'] = df['clean_review'].apply(len)
df['word_count'] = df['clean_review'].apply(lambda x: len(x.split()))

In [33]:
df['rating_category'] = df['rating'].apply(
    lambda x: 'low' if x <= 4 else ('medium' if x <= 7 else 'high')
)

In [34]:
df.head()

,drugName,condition,review,rating,date,usefulCount,clean_review,year,month,review_length,word_count,rating_category
0,Cyclosporine,keratoconjunctivitis sicca,"""i have used restasis for about a year now and...",2.0,2013-04-20,69,i have used restasis for about a year now and ...,2013,4,724,142,low
1,Etonogestrel,birth control,"""my experience has been somewhat mixed. i have...",7.0,2016-08-07,4,my experience has been somewhat mixed i have b...,2016,8,716,134,medium
2,Implanon,birth control,"""this is my second implanon would not recommen...",1.0,2016-05-11,6,this is my second implanon would not recommend...,2016,5,713,138,low
3,Hydroxyzine,anxiety,"""i recommend taking as prescribed, and the bot...",10.0,2012-03-19,124,i recommend taking as prescribed and the bottl...,2012,3,518,102,high
4,Dalfampridine,multiple sclerosis,"""i have been on ampyra for 5 days and have bee...",9.0,2010-08-01,101,i have been on ampyra for days and have been ...,2010,8,361,72,high


In [36]:
df.to_csv("../data/clean_drug_reviews.csv", index=False)